# Social Media Content Performance Data Pipeline

## 1. Imports Library & Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
INPUT_PATH = r"C:\Users\ADMIN\OneDrive - Hanoi University of Science and Technology\Big_Project_Social_Media_Content_Performance_Dataset.csv"  # update before running

# Khai báo thư mục đầu ra để lưu các file kết quả của pipeline
OUT_DIR = Path('pipeline_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Tên cột dùng để nhận diện nền tảng social media
PLATFORM_COL = 'Platform'

# Tên cột khóa dùng để định danh một post duy nhất
KEY_COL = 'Post_key'

# Tên cột chứa engagement rate gốc trong dataset
ENGAGEMENT_RATE_COL = 'Engagement_Rate'

## 2. Cleaning & Standardization functions

In [ ]:
def standardize_rate(value):
    # Nếu giá trị bị thiếu (NaN) thì giữ nguyên là NaN
    if pd.isna(value):
        return np.nan

    # Ép mọi giá trị về chuỗi và loại bỏ khoảng trắng đầu cuối để xử lý nhất quán cả số, text, và giá trị khác
    text = str(value).strip()

    # Các giá trị rỗng hoặc biểu diễn missing phổ biến sẽ được quy về NaN để tránh lỗi khi convert
    if text == "" or text.lower() in {"nan", "nat", "none", "null"}:
        return np.nan

    # Nếu dữ liệu có ký hiệu %, ví dụ "10%" thì bỏ dấu % và chia 100 để đưa về dạng decimal: 0.1
    if text.endswith("%"):
        try:
            return float(text[:-1].replace(",", ".")) / 100.0
        except ValueError:
            # Nếu không convert được thì trả về NaN thay vì làm code bị vỡ
            return np.nan

    # Nếu không có %, thử convert trực tiếp sang float đồng thời thay dấu phẩy bằng dấu chấm để hỗ trợ format như "10,5"
    try:
        return float(text.replace(",", "."))
    except ValueError:
        # Nếu vẫn lỗi thì coi như dữ liệu không hợp lệ
        return np.nan


def make_raw_clean(df_in, engagement_rate_col="Engagement_Rate"):
    # Tạo bản copy để không chỉnh trực tiếp dataframe gốc
    df = df_in.copy()

    # Danh sách các cột đáng ra phải là số cần convert về dạng số để phục vụ tính toán/QA/phân tích
    numeric_cols = ["Clicks", "Click_Through_Rate", "Impressions", "Views", "Engagement",
                    "Likes", "Comments", "Shares"]

    # Với mỗi cột số, nếu cột đó tồn tại trong dữ liệu thì đưa về kiểu numeric, giá trị lỗi sẽ thành NaN
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Nếu có cột Engagement_Rate thì chuẩn hóa về dạng số thập phân và lưu ở cột mới engagement_rate_num để không làm mất dữ liệu gốc
    if engagement_rate_col in df.columns:
        df["engagement_rate_num"] = df[engagement_rate_col].apply(standardize_rate)

    # Kiểm tra xem dữ liệu có đủ cả 2 cột dùng để xác định click-tracking hay không
    has_click_cols = {"Clicks", "Click_Through_Rate"}.issubset(df.columns)

    if has_click_cols:
        # Một dòng được xem là có click data khi cả Clicks và CTR đều không bị thiếu
        df["Has_click_data"] = df["Clicks"].notna() & df["Click_Through_Rate"].notna()
    else:
        # Nếu dataset không có đủ 2 cột này thì gán toàn bộ là False
        df["Has_click_data"] = False

    # Tạo cột phân loại phạm vi theo dõi traffic:
    # - "Tracked" nếu có click data
    # - "Not tracked" nếu không có
    df["Traffic_scope"] = np.where(df["Has_click_data"], "Tracked", "Not tracked")

    # Trả về bảng raw_clean đã được chuẩn hóa cơ bản
    return df


def make_stg_dedup(df_clean, key_col="Post_key"):
    # Đảm bảo bảng đầu vào có cột khóa dùng để xác định 1 post duy nhất
    if key_col not in df_clean.columns:
        raise ValueError(f"Missing required key column: {key_col}")

    # Xác định thứ tự ưu tiên để chọn bản ghi tốt nhất khi trùng Post_key:
    # 1. Ưu tiên dòng có click data
    # 2. Sau đó ưu tiên dòng có impressions cao hơn
    # 3. Rồi engagement cao hơn
    # 4. Rồi views cao hơn
    # 5. Cuối cùng là clicks cao hơn
    sort_cols = ["Has_click_data", "Impressions", "Engagement", "Views", "Clicks"]

    # Chỉ giữ lại những cột thực sự tồn tại trong dataframe để tránh lỗi nếu dataset không có đủ mọi cột
    sort_cols = [col for col in sort_cols if col in df_clean.columns]

    return (
        # Sắp xếp giảm dần theo các tiêu chí ưu tiên
        df_clean.sort_values(sort_cols, ascending=[False] * len(sort_cols))

        # Sau khi sắp xếp, nếu có nhiều dòng cùng Post_key thì giữ dòng đầu tiên tương đương với dòng tốt nhất theo thứ tự ưu tiên ở trên
        .drop_duplicates(subset=[key_col], keep="first")
        .reset_index(drop=True)
    )

## 3. Load raw data

In [7]:
df_raw = pd.read_csv(INPUT_PATH)
print('Loaded shape:', df_raw.shape)
df_raw.head()

Loaded shape: (5600, 29)


,Post_ID,Platform,Content_Type,Content_Category,Post_Type,Region,Longitude,Latitude,Engagement,Views,...,Has_click_data,Traffic_scope,Main_Hashtag,Post_Published_At,Post_Date,Post_Hour,Engagement_Level,Post_Hour_Clean,Post_Date_Clean,Post_key
0,Post_1,TikTok,Organic,Product Promotion,Video,UK,-3.4360,55.3781,54510,599342,...,True,Tracked,#FeatureHighlight,2024-03-14 15:42:00,2024-03-14 00:00:00,15,Low,3:42:00 PM,3/14/2024,Post_1|TIKTOK|2024-03-14T15:42:00
1,Post_2,Instagram,Organic,Product Promotion,Video,India,78.9629,20.5937,259440,1896301,...,False,Not tracked,#ProductDemo,2024-04-24 12:02:00,2024-04-24 00:00:00,12,Medium,12:02:00 PM,4/24/2024,Post_2|INSTAGRAM|2024-04-24T12:02:00
2,Post_3,X.com,Organic,Entertainment,Text,Brazil,-51.9253,-14.2350,14182,253052,...,False,Not tracked,#MemeMonday,2024-12-01 10:02:00,2024-12-01 00:00:00,10,Low,10:02:00 AM,12/1/2024,Post_3|X.COM|2024-12-01T10:02:00
3,Post_4,Instagram,Organic,Entertainment,Carousel,Australia,133.7751,-25.2744,49137,735640,...,False,Not tracked,#JustForFun,2024-07-07 12:59:00,2024-07-07 00:00:00,12,Low,12:59:00 PM,7/7/2024,Post_4|INSTAGRAM|2024-07-07T12:59:00
4,Post_5,TikTok,Organic,Educational,Video,Brazil,-51.9253,-14.2350,20832,77358,...,True,Tracked,#CaseStudy2025,2024-01-10 11:28:00,2024-01-10 00:00:00,11,High,11:28:00 AM,1/10/2024,Post_5|TIKTOK|2024-01-10T11:28:00


## 4.Pipeline layers

In [ ]:
# Gọi hàm make_raw_clean để tạo bảng RAW_clean từ dữ liệu gốc
# Bước này thực hiện các xử lý làm sạch cơ bản như:
# - gán kiểu các cột số
# - chuẩn hóa Engagement_Rate
# - tạo cột Has_click_data và Traffic_scope
df_raw_clean = make_raw_clean(df_raw, engagement_rate_col=ENGAGEMENT_RATE_COL)

# Gọi hàm make_stg_dedup để tạo bảng STG_dedup từ RAW_clean
# Bước này loại bỏ các bản ghi trùng Post_key và giữ lại dòng tốt nhất
# theo thứ tự ưu tiên đã định trong hàm
df_stg_dedup = make_stg_dedup(df_raw_clean, key_col=KEY_COL)

print('RAW_clean shape:', df_raw_clean.shape)
print('STG_dedup shape:', df_stg_dedup.shape)

# Đếm số dòng có Post_key bị trùng trong RAW_clean
print('Duplicate Post_key rows in RAW_clean:', df_raw_clean[KEY_COL].duplicated().sum())

# Đếm số dòng có Post_key bị trùng còn lại trong STG_dedup
print('Duplicate Post_key rows in STG_dedup:', df_stg_dedup[KEY_COL].duplicated().sum())

RAW_clean shape: (5600, 30)
STG_dedup shape: (5579, 30)
Duplicate Post_key rows in RAW_clean: 21
Duplicate Post_key rows in STG_dedup: 0


## 5. Quality Assurance outputs

In [ ]:
# Tạo bảng tóm tắt QA tổng quan cho 2 layer:
# - RAW_clean: dữ liệu sau khi làm sạch cơ bản
# - STG_dedup: dữ liệu sau khi loại trùng theo Post_key
qa_overview = pd.DataFrame({
    "metric": [
        "rows_raw_clean",                 # Số dòng của bảng RAW_clean
        "rows_stg_dedup",                 # Số dòng của bảng STG_dedup
        "dup_post_key_raw_clean",         # Số Post_key bị trùng ở RAW_clean
        "dup_post_key_stg_dedup",         # Số Post_key bị trùng còn lại ở STG_dedup
        "has_click_data_rate_raw_clean",  # Tỷ lệ dòng có đủ dữ liệu click-tracking ở RAW_clean
        "has_click_data_rate_stg_dedup"   # Tỷ lệ dòng có đủ dữ liệu click-tracking ở STG_dedup
    ],
    "value": [
        len(df_raw_clean),  # Đếm tổng số dòng của RAW_clean

        len(df_stg_dedup),  # Đếm tổng số dòng của STG_dedup

        # Đếm số Post_key bị trùng trong RAW_clean
        int(df_raw_clean[KEY_COL].duplicated().sum()),

        # Kiểm tra sau dedup thì còn bao nhiêu Post_key bị trùng
        int(df_stg_dedup[KEY_COL].duplicated().sum()),

        # Tính tỷ lệ dòng có click data trong RAW_clea
        float(df_raw_clean["Has_click_data"].mean()),

        # Tính tỷ lệ dòng có click data trong STG_dedup
        float(df_stg_dedup["Has_click_data"].mean())
    ]
})

# Tạo bảng QA về missingness của từng cột trong STG_dedup
qa_missing = (df_stg_dedup.isna().mean() * 100).sort_values(ascending=False).reset_index()

# Đổi tên cột cho rõ nghĩa hơn
qa_missing.columns = ["column", "missing_rate_percent"]

# Tính tỷ lệ thiếu dữ liệu click-tracking theo từng platform
# Mục tiêu: xem platform nào hay bị thiếu Clicks hoặc CTR hơn platform khác
qa_missing_by_platform = (
    df_stg_dedup.groupby(PLATFORM_COL)[["Clicks", "Click_Through_Rate"]]

    # Với mỗi platform, tính tỷ lệ thiếu của 2 cột Clicks và CTR
    .apply(lambda group: group.isna().mean() * 100)

    .reset_index()
)

qa_missing_by_platform.columns = [PLATFORM_COL, "Clicks_missing_rate_percent", "CTR_missing_rate_percent"]

# Lấy mẫu các dòng bị trùng Post_key trong RAW_clean để kiểm tra thủ công
qa_duplicate_sample = (
    df_raw_clean[df_raw_clean[KEY_COL].duplicated(keep=False)]
    .sort_values(KEY_COL)
    .head(20)
)

# Hiển thị bảng QA tổng quan
#qa_overview
#qa_missing
#qa_missing_by_platform
qa_duplicate_sample


,Post_ID,Platform,Content_Type,Content_Category,Post_Type,Region,Longitude,Latitude,Engagement,Views,...,Traffic_scope,Main_Hashtag,Post_Published_At,Post_Date,Post_Hour,Engagement_Level,Post_Hour_Clean,Post_Date_Clean,Post_key,engagement_rate_num
5289,Post_1021,Facebook,Sponsored,Customer Story,Image,USA,-95.7129,37.0902,56466,257877,...,Tracked,#Testimonial,2024-03-08 17:37:00,2024-03-08 00:00:00,17,High,5:37:00 PM,3/8/2024,Post_1021|FACEBOOK|2024-03-08T17:37:00,0.218965
5443,Post_1021,Facebook,Organic,Educational,Video,USA,-95.7129,37.0902,56966,257877,...,Tracked,#CustomerStory,2024-03-08 17:37:00,2024-03-08 00:00:00,17,High,5:37:00 PM,3/8/2024,Post_1021|FACEBOOK|2024-03-08T17:37:00,0.220908
5102,Post_1420,LinkedIn,Organic,Entertainment,Article,India,78.9629,20.5937,44654,767331,...,Tracked,#DidYouKnow,10:19:00 AM,3/15/2025,10,Low,10:19:00 AM,3/15/2025,Post_1420|LINKEDIN|2025-03-15T10:19:00,0.060000
1406,Post_1420,LinkedIn,Organic,Entertainment,Text,India,78.9629,20.5937,43722,767331,...,Not tracked,#FunContent,10:19:00 AM,3/15/2025,10,Low,10:19:00 AM,3/15/2025,Post_1420|LINKEDIN|2025-03-15T10:19:00,0.060000
5220,Post_150,LinkedIn,Organic,Customer Story,Article,UK,-3.4360,55.3781,36702,207311,...,Tracked,#CustomerSuccess,11:24:00 AM,3/18/2025,11,Medium,11:24:00 AM,3/18/2025,Post_150|LINKEDIN|2025-03-18T11:24:00,0.180000
146,Post_150,LinkedIn,Organic,Customer Story,Image,UK,-3.4360,55.3781,35214,207311,...,Not tracked,#SuccessStory,11:24:00 AM,3/18/2025,11,Medium,11:24:00 AM,3/18/2025,Post_150|LINKEDIN|2025-03-18T11:24:00,0.170000
5116,Post_152,LinkedIn,Organic,Educational,Article,USA,-95.7129,37.0902,81648,416902,...,Tracked,#DidYouKnow,1:12:00 PM,12/3/2024,13,Medium,1:12:00 PM,12/3/2024,Post_152|LINKEDIN|2024-12-03T13:12:00,0.200000
148,Post_152,LinkedIn,Organic,Customer Story,Image,USA,-95.7129,37.0902,67447,416902,...,Not tracked,#SuccessStory,1:12:00 PM,12/3/2024,13,Medium,1:12:00 PM,12/3/2024,Post_152|LINKEDIN|2024-12-03T13:12:00,0.160000
1792,Post_1808,LinkedIn,Sponsored,Entertainment,Video,Japan,138.2529,36.2048,27204,517391,...,Not tracked,#FunContent,3:12:00 PM,4/25/2024,15,Low,3:12:00 PM,4/25/2024,Post_1808|LINKEDIN|2024-04-25T15:12:00,0.050000
5123,Post_1808,LinkedIn,Sponsored,Entertainment,Article,Japan,138.2529,36.2048,33244,517391,...,Tracked,#DidYouKnow,3:12:00 PM,4/25/2024,15,Low,3:12:00 PM,4/25/2024,Post_1808|LINKEDIN|2024-04-25T15:12:00,0.060000


## 6. Save outputs

In [ ]:
# xuất các File
(OUT_DIR / "RAW_clean.csv").parent.mkdir(parents=True, exist_ok=True)

df_raw_clean.to_csv(OUT_DIR / "RAW_clean.csv", index=False)

df_stg_dedup.to_csv(OUT_DIR / "STG_dedup.csv", index=False)

qa_overview.to_csv(OUT_DIR / "QA_overview.csv", index=False)

qa_missing.to_csv(OUT_DIR / "QA_missingness_stg.csv", index=False)

qa_missing_by_platform.to_csv(OUT_DIR / "QA_missing_by_platform_stg.csv", index=False)

qa_duplicate_sample.to_csv(OUT_DIR / "QA_duplicate_post_key_sample_raw.csv", index=False)

list(OUT_DIR.glob("*.csv"))

[WindowsPath('pipeline_outputs/QA_duplicate_post_key_sample_raw.csv'),
 WindowsPath('pipeline_outputs/QA_missingness_stg.csv'),
 WindowsPath('pipeline_outputs/QA_missing_by_platform_stg.csv'),
 WindowsPath('pipeline_outputs/QA_overview.csv'),
 WindowsPath('pipeline_outputs/RAW_clean.csv'),
 WindowsPath('pipeline_outputs/STG_dedup.csv')]

## 7. BigQuery load


In [ ]:
from google.cloud import bigquery
from google.oauth2 import service_account
from pathlib import Path

project_id = "jda-k1"
dataset_id = "practice_data_pipeline"
key_path =r"C:\Users\ADMIN\Downloads\jda-k1-2afad6d8f47e.json"


credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

DATA_DIR = Path("pipeline_outputs")
raw_csv = DATA_DIR / "RAW_clean.csv"
stg_csv = DATA_DIR / "STG_dedup.csv"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

def load_csv(csv_path, table_name):
    table_ref = f"{project_id}.{dataset_id}.{table_name}"
    print("Loading", csv_path.name, "->", table_ref)
    with open(csv_path, "rb") as f:
        job = client.load_table_from_file(f, table_ref, job_config=job_config)
    job.result()
    print("Done. Rows:", client.get_table(table_ref).num_rows)

load_csv(raw_csv, "social_raw_clean")
load_csv(stg_csv, "social_stg_dedup")